# Task 2: SQL for Data Extraction & Analysis

> **ApexPlanet Data Analytics Internship**  
> *30-Day Program | Day 7-13*

## What You'll Learn
- Day 7-8: SQL Fundamentals (SELECT, WHERE, GROUP BY, JOIN)
- Day 9-10: Advanced SQL (Subqueries, CTEs, Window Functions)
- Day 11-13: Python + SQL Integration + 10 Business Questions

---

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Set style for charts
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")

## Step 1: Create SQLite Database

SQLite is a lightweight database built into Python. No installation needed!

In [ ]:
# Connect to SQLite database (creates file if doesn't exist)
conn = sqlite3.connect('../database/superstore.db')

# Load cleaned data from Task 1
df = pd.read_csv('../data/superstore_cleaned.csv')

# Save to database as 'sales' table
df.to_sql('sales', conn, if_exists='replace', index=False)

print(f"✅ Database created with {len(df)} rows")
print(f"✅ Table 'sales' created successfully")

## Step 2: Basic SQL Queries (Day 7-8)

### Query 1: SELECT - View all data

In [ ]:
# Select all columns, limit to 5 rows
query1 = "SELECT * FROM sales LIMIT 5"
result1 = pd.read_sql(query1, conn)
print("Query: SELECT * FROM sales LIMIT 5")
print("\nResult:")
print(result1)

### Query 2: WHERE - Filter data

Show only high-value orders (Sales > $15,000)

In [ ]:
query2 = """
SELECT Order_ID, Customer_Name, Category, Sales, Profit 
FROM sales 
WHERE Sales > 15000
ORDER BY Sales DESC
LIMIT 10
"""
result2 = pd.read_sql(query2, conn)
print("Query: Orders with Sales > $15,000")
print(result2)

### Query 3: GROUP BY - Aggregate data

Summary statistics by Category

In [ ]:
query3 = """
SELECT 
    Category,
    COUNT(*) as Total_Orders,
    SUM(Sales) as Total_Sales,
    AVG(Sales) as Avg_Sales,
    SUM(Profit) as Total_Profit
FROM sales 
GROUP BY Category
ORDER BY Total_Sales DESC
"""
result3 = pd.read_sql(query3, conn)
print("Query: Summary by Category")
print(result3)

### Query 4: JOIN - Combine tables

Create a lookup table and join with sales data

In [ ]:
# Create region managers lookup table
region_managers = pd.DataFrame({
    'Region': ['West', 'East', 'Central', 'South'],
    'Manager': ['Sarah Johnson', 'Mike Chen', 'Lisa Rodriguez', 'David Park'],
    'Target_Sales': [15000000, 14000000, 13000000, 13500000]
})
region_managers.to_sql('region_managers', conn, if_exists='replace', index=False)

# JOIN query
query4 = """
SELECT 
    s.Region,
    rm.Manager,
    rm.Target_Sales,
    SUM(s.Sales) as Actual_Sales,
    ROUND((SUM(s.Sales) / rm.Target_Sales) * 100, 2) as Achievement_Pct
FROM sales s
INNER JOIN region_managers rm ON s.Region = rm.Region
GROUP BY s.Region, rm.Manager, rm.Target_Sales
ORDER BY Actual_Sales DESC
"""
result4 = pd.read_sql(query4, conn)
print("Query: Regional Performance vs Targets")
print(result4)

## Step 3: Advanced SQL (Day 9-10)

### Query 5: Subquery

In [ ]:
# Subquery: Orders above average sales
query5 = """
SELECT Order_ID, Customer_Name, Category, Sales, Profit
FROM sales
WHERE Sales > (SELECT AVG(Sales) FROM sales)
ORDER BY Sales DESC
LIMIT 10
"""
result5 = pd.read_sql(query5, conn)
print("Query: Orders above average sales")
print(result5)

### Query 6: CTE (Common Table Expression)

In [ ]:
query6 = """
WITH monthly_summary AS (
    SELECT 
        Order_Year, Order_Month, Order_Month_Name,
        COUNT(*) as Total_Orders,
        SUM(Sales) as Total_Sales,
        SUM(Profit) as Total_Profit
    FROM sales
    GROUP BY Order_Year, Order_Month, Order_Month_Name
)
SELECT 
    Order_Year, Order_Month_Name, Total_Orders,
    ROUND(Total_Sales, 2) as Total_Sales,
    ROUND(Total_Profit, 2) as Total_Profit,
    ROUND((Total_Profit / Total_Sales) * 100, 2) as Profit_Margin_Pct
FROM monthly_summary
ORDER BY Total_Sales DESC
LIMIT 10
"""
result6 = pd.read_sql(query6, conn)
print("Query: Monthly Performance (CTE)")
print(result6)

### Query 7: Window Functions

In [ ]:
query7 = """
SELECT * FROM (
    SELECT 
        Order_ID, Customer_Name, Category, Sales, Profit,
        ROW_NUMBER() OVER (PARTITION BY Category ORDER BY Sales DESC) as Rank_in_Category
    FROM sales
) ranked
WHERE Rank_in_Category <= 3
ORDER BY Category, Rank_in_Category
"""
result7 = pd.read_sql(query7, conn)
print("Query: Top 3 in each category (Window Function)")
print(result7)

## Step 4: 10 Business Questions with Visualizations (Day 11-13)

### Q1: Top 5 Products by Sales

In [ ]:
q1 = pd.read_sql("""
SELECT Product_Name, Category, SUM(Sales) as Total_Sales,
SUM(Profit) as Total_Profit, COUNT(*) as Times_Ordered
FROM sales GROUP BY Product_Name, Category
ORDER BY Total_Sales DESC LIMIT 5
""", conn)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(q1['Product_Name'], q1['Total_Sales'], color='#3498DB')
ax.set_xlabel('Total Sales ($)')
ax.set_title('Q1: Top 5 Products by Sales', fontweight='bold', fontsize=14)
ax.invert_yaxis()
for i, v in enumerate(q1['Total_Sales']):
    ax.text(v + 5000, i, f'${v:,.0f}', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../images/task2/q1_top_products.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved: q1_top_products.png")

### Q2: Monthly Sales Trend

In [ ]:
q2 = pd.read_sql("""
SELECT Order_Year || '-' || printf('%02d', Order_Month) as Year_Month,
Order_Year, Order_Month, Order_Month_Name,
SUM(Sales) as Total_Sales, SUM(Profit) as Total_Profit, COUNT(*) as Order_Count
FROM sales GROUP BY Order_Year, Order_Month, Order_Month_Name
ORDER BY Order_Year, Order_Month
""", conn)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(q2['Year_Month'], q2['Total_Sales'], marker='o', linewidth=2, color='#E74C3C')
ax.fill_between(range(len(q2)), q2['Total_Sales'], alpha=0.2, color='#E74C3C')
ax.set_xlabel('Year-Month')
ax.set_ylabel('Total Sales ($)')
ax.set_title('Q2: Monthly Sales Trend (2014-2017)', fontweight='bold', fontsize=14)
ax.tick_params(axis='x', rotation=45)
ax.set_xticks(range(0, len(q2), 6))
ax.set_xticklabels(q2['Year_Month'].iloc[::6])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../images/task2/q2_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

### Q3: Customer Segmentation by Spend

In [ ]:
q3 = pd.read_sql("""
SELECT Customer_Name, Segment, COUNT(*) as Total_Orders,
SUM(Sales) as Total_Spent, AVG(Sales) as Avg_Order_Value, SUM(Profit) as Total_Profit
FROM sales GROUP BY Customer_Name, Segment ORDER BY Total_Spent DESC LIMIT 15
""", conn)

seg_data = pd.read_sql("SELECT Segment, SUM(Sales) as Total_Sales FROM sales GROUP BY Segment", conn)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].pie(seg_data['Total_Sales'], labels=seg_data['Segment'], autopct='%1.1f%%',
            colors=['#3498DB', '#E74C3C', '#2ECC71'], startangle=90)
axes[0].set_title('Sales by Segment', fontweight='bold')
axes[1].barh(q3['Customer_Name'].head(10), q3['Total_Spent'].head(10), color='#9B59B6')
axes[1].set_xlabel('Total Spent ($)')
axes[1].set_title('Top 10 Customers by Spend', fontweight='bold')
axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig('../images/task2/q3_customer_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

### Q4: Regional Performance vs Targets

In [ ]:
q4 = pd.read_sql("""
SELECT s.Region, rm.Manager, rm.Target_Sales, SUM(s.Sales) as Actual_Sales,
SUM(s.Profit) as Total_Profit, ROUND((SUM(s.Sales) / rm.Target_Sales) * 100, 2) as Achievement_Pct
FROM sales s INNER JOIN region_managers rm ON s.Region = rm.Region
GROUP BY s.Region, rm.Manager, rm.Target_Sales ORDER BY Achievement_Pct DESC
""", conn)

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(q4))
width = 0.35
ax.bar([i - width/2 for i in x], q4['Target_Sales'], width, label='Target', color='#3498DB', alpha=0.7)
ax.bar([i + width/2 for i in x], q4['Actual_Sales'], width, label='Actual', color='#2ECC71', alpha=0.7)
ax.set_xlabel('Region')
ax.set_ylabel('Sales ($)')
ax.set_title('Q4: Regional Performance vs Targets', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(q4['Region'])
ax.legend()
plt.tight_layout()
plt.savefig('../images/task2/q4_regional_targets.png', dpi=150, bbox_inches='tight')
plt.show()

### Q5: Profit Margin by Sub-Category

In [ ]:
q5 = pd.read_sql("""
SELECT Category, Sub_Category, SUM(Sales) as Total_Sales, SUM(Profit) as Total_Profit,
ROUND((SUM(Profit) / SUM(Sales)) * 100, 2) as Profit_Margin_Pct, COUNT(*) as Order_Count
FROM sales GROUP BY Category, Sub_Category ORDER BY Profit_Margin_Pct DESC
""", conn)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#2ECC71' if v > 0 else '#E74C3C' for v in q5['Profit_Margin_Pct']]
ax.barh(q5['Sub_Category'], q5['Profit_Margin_Pct'], color=colors)
ax.set_xlabel('Profit Margin (%)')
ax.set_title('Q5: Profit Margin by Sub-Category', fontweight='bold', fontsize=14)
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../images/task2/q5_profit_margin.png', dpi=150, bbox_inches='tight')
plt.show()

### Q6: Discount Impact on Profitability

In [ ]:
q6 = pd.read_sql("""
SELECT CASE WHEN Discount = 0 THEN 'No Discount'
WHEN Discount <= 0.1 THEN 'Low Discount (1-10%)'
WHEN Discount <= 0.2 THEN 'Medium Discount (11-20%)'
ELSE 'High Discount (>20%)' END as Discount_Level,
COUNT(*) as Order_Count, AVG(Sales) as Avg_Sales,
AVG(Profit) as Avg_Profit, SUM(Profit) as Total_Profit
FROM sales GROUP BY Discount_Level ORDER BY Avg_Profit DESC
""", conn)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ECC71', '#F1C40F', '#E67E22', '#E74C3C']
ax.bar(q6['Discount_Level'], q6['Avg_Profit'], color=colors)
ax.set_ylabel('Average Profit ($)')
ax.set_title('Q6: Average Profit by Discount Level', fontweight='bold', fontsize=14)
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('../images/task2/q6_discount_impact.png', dpi=150, bbox_inches='tight')
plt.show()

### Q7: Shipping Mode Efficiency

In [ ]:
q7 = pd.read_sql("""
SELECT Ship_Mode, COUNT(*) as Total_Orders, AVG(Shipping_Days) as Avg_Shipping_Days,
SUM(Sales) as Total_Sales, AVG(Profit) as Avg_Profit
FROM sales GROUP BY Ship_Mode ORDER BY Avg_Shipping_Days
""", conn)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(q7['Ship_Mode'], q7['Avg_Shipping_Days'], color=['#16A085', '#27AE60', '#2980B9', '#8E44AD'])
ax.set_ylabel('Average Shipping Days')
ax.set_title('Q7: Shipping Mode Efficiency', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../images/task2/q7_shipping_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

### Q8: Quarterly Performance

In [ ]:
q8 = pd.read_sql("""
SELECT Order_Year, Order_Quarter, 'Q' || Order_Quarter as Quarter_Name,
COUNT(*) as Total_Orders, SUM(Sales) as Total_Sales, SUM(Profit) as Total_Profit,
ROUND((SUM(Profit) / SUM(Sales)) * 100, 2) as Profit_Margin_Pct
FROM sales GROUP BY Order_Year, Order_Quarter ORDER BY Order_Year, Order_Quarter
""", conn)

fig, ax = plt.subplots(figsize=(12, 6))
quarters = [f"{row['Order_Year']}{row['Quarter_Name']}" for _, row in q8.iterrows()]
ax.plot(quarters, q8['Total_Sales'], marker='o', linewidth=2, color='#9B59B6')
ax.fill_between(range(len(quarters)), q8['Total_Sales'], alpha=0.2, color='#9B59B6')
ax.set_xlabel('Quarter')
ax.set_ylabel('Total Sales ($)')
ax.set_title('Q8: Quarterly Sales Performance', fontweight='bold', fontsize=14)
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../images/task2/q8_quarterly_performance.png', dpi=150, bbox_inches='tight')
plt.show()

### Q9: Loss-Making Orders

In [ ]:
loss_by_cat = pd.read_sql("""
SELECT Category, COUNT(*) as Loss_Count, SUM(Profit) as Total_Loss
FROM sales WHERE Profit < 0 GROUP BY Category
""", conn)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(loss_by_cat['Category'], loss_by_cat['Loss_Count'], color=['#E74C3C', '#E67E22', '#F39C12'])
ax.set_ylabel('Number of Loss-Making Orders')
ax.set_title('Q9: Loss-Making Orders by Category', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../images/task2/q9_loss_orders.png', dpi=150, bbox_inches='tight')
plt.show()

### Q10: Year-over-Year Growth

In [ ]:
q10 = pd.read_sql("""
WITH yearly_sales AS (
    SELECT Order_Year, SUM(Sales) as Total_Sales,
    SUM(Profit) as Total_Profit, COUNT(*) as Total_Orders
    FROM sales GROUP BY Order_Year
)
SELECT Order_Year, Total_Sales, Total_Profit, Total_Orders,
LAG(Total_Sales, 1) OVER (ORDER BY Order_Year) as Prev_Year_Sales,
ROUND(((Total_Sales - LAG(Total_Sales, 1) OVER (ORDER BY Order_Year)) 
/ LAG(Total_Sales, 1) OVER (ORDER BY Order_Year)) * 100, 2) as YoY_Growth_Pct
FROM yearly_sales ORDER BY Order_Year
""", conn)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(q10['Order_Year'], q10['Total_Sales'], color='#3498DB', alpha=0.7, label='Total Sales')
ax2 = ax.twinx()
ax2.plot(q10['Order_Year'], q10['YoY_Growth_Pct'], color='#E74C3C', marker='o',
         linewidth=2, markersize=8, label='YoY Growth %')
ax.set_xlabel('Year')
ax.set_ylabel('Total Sales ($)', color='#3498DB')
ax2.set_ylabel('YoY Growth (%)', color='#E74C3C')
ax.set_title('Q10: Year-over-Year Sales & Growth', fontweight='bold', fontsize=14)
ax.tick_params(axis='y', labelcolor='#3498DB')
ax2.tick_params(axis='y', labelcolor='#E74C3C')
plt.tight_layout()
plt.savefig('../images/task2/q10_yoy_growth.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

This notebook demonstrated:
- ✅ SQLite database creation
- ✅ Basic SQL queries (SELECT, WHERE, GROUP BY, JOIN)
- ✅ Advanced SQL (Subqueries, CTEs, Window Functions)
- ✅ Python + SQL integration with pandas
- ✅ 10 business questions answered with visualization

**All charts saved to:** `../images/task2/`

In [ ]:
# Close database connection
conn.close()
print("✅ Database connection closed. Task 2 complete!")